In [1]:
import numpy as np
import pandas as np

import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
import pandas as pd
import numpy as np

# =========================
# 경로 설정 (비워둠)
# =========================
DF2_PATH = r"C:/Users/qkrtl/Downloads/df2.csv"                  # 예: r"C:\...\df2.csv"
MASTER_PATH = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master_all_columns.csv"               # 예: r"C:\...\master_all_columns.csv"


# =========================
# 조인 키 (기본: matchId + name)
# =========================
KEYS = ["matchId", "name"]

# survival_time 컬럼 후보 (master에서 찾기)
SURVIVAL_CANDIDATES = ["survival_time", "survivalTime", "timeAlive", "TimeAlive"]
SURVIVAL_THRESHOLD = 120  # 2분

# =========================
# 유틸
# =========================
def normalize_key_values(df: pd.DataFrame, keys):
    out = df.copy()
    for k in keys:
        out[k] = out[k].astype("string").str.strip().replace("", pd.NA)
    return out

def pick_survival_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# =========================
# 1) 로드
# =========================
df2 = pd.read_csv(DF2_PATH, low_memory=False)
master = pd.read_csv(MASTER_PATH, low_memory=False)

# 키 존재 체크
for k in KEYS:
    if k not in df2.columns:
        raise KeyError(f"df2에 키 컬럼이 없습니다: {k}")
    if k not in master.columns:
        raise KeyError(f"master_all_columns에 키 컬럼이 없습니다: {k}")

# 키 정규화(공백/타입)
df2_n = normalize_key_values(df2, KEYS)
master_n = normalize_key_values(master, KEYS)

# =========================
# 2) master 쪽 중복키 처리 (중요)
#    - left join에서 master에 같은 키가 여러 개면 df2 행이 증식함
# =========================
dup_master = master_n.duplicated(KEYS).sum()
print(f"[master] duplicated keys rows = {dup_master:,}")

if dup_master > 0:
    # 기본 정책: 같은 키면 "첫 번째"만 사용
    # (원하면 '마지막' 또는 특정 컬럼 기준 정렬 후 drop_duplicates로 바꿔도 됨)
    master_n = master_n.drop_duplicates(KEYS, keep="first").copy()
    print(f"[master] after dedup, rows = {len(master_n):,}")

# =========================
# 3) df2 기준 LEFT JOIN
#    - 겹치는 컬럼명은 _df2 / _master suffix로 구분
# =========================
merged = df2_n.merge(
    master_n,
    on=KEYS,
    how="left",
    suffixes=("_df2", "_master"),
    indicator=True,            # 매칭 여부 확인용
    validate="m:1"             # df2 many : master one (dedup 했으므로 이게 정상)
)

# =========================
# 4) 매칭 리포트
# =========================
vc = merged["_merge"].value_counts(dropna=False)
print("\n[JOIN 결과 _merge 카운트]")
print(vc.to_string())

match_rate = (merged["_merge"] == "both").mean()
print(f"\n[JOIN 매칭률] {match_rate:.2%} (df2 기준)")

# =========================
# 5) survival_time 기반 플래그 컬럼 생성 (필터 대신 컬럼으로 관리)
# =========================
surv_col = pick_survival_col(merged, SURVIVAL_CANDIDATES)
print(f"\n[사용 survival 컬럼] {surv_col}")

if surv_col is not None:
    merged[surv_col] = pd.to_numeric(merged[surv_col], errors="coerce")
    merged["is_2min_over"] = merged[surv_col].ge(SURVIVAL_THRESHOLD)
    merged["is_2min_under"] = merged[surv_col].lt(SURVIVAL_THRESHOLD)
    merged["survival_is_na"] = merged[surv_col].isna()
else:
    # master에서 survival_time이 안 붙은 경우(매칭 실패 포함) 대비
    merged["is_2min_over"] = pd.NA
    merged["is_2min_under"] = pd.NA
    merged["survival_is_na"] = pd.NA

# =========================
# 6) 필요하면 indicator 컬럼 제거
# =========================
# merged = merged.drop(columns=["_merge"])

# =========================
# 7) 저장(원하면)
# =========================
OUT_PATH = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master.csv" 
merged.to_csv(OUT_PATH, index=False)
print("\n완료: df2 LEFT JOIN master_all_columns + is_2min_over 생성")


[master] duplicated keys rows = 28,045
[master] after dedup, rows = 823,389

[JOIN 결과 _merge 카운트]
_merge
both          821596
left_only       6505
right_only         0

[JOIN 매칭률] 99.21% (df2 기준)

[사용 survival 컬럼] survival_time

완료: df2 LEFT JOIN master_all_columns + is_2min_over 생성


In [8]:
df = pd.read_csv("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master.csv")
df.head()


,matchId,mapName_df2,gameMode_df2,createdAt_df2,DBNOs_df2,assists_df2,boosts,damageDealt,deathType_df2,headshotKills_df2,...,weaponsAcquired_master,winPlace_df2,servername_master,currentRankPoint_master,game_type_master,tier_master,_merge,is_2min_over,is_2min_under,survival_is_na
0,6f3b4b92-5c17-4a7e-a8b0-4a35ad7c1d7a,Desert_Main,squad,2026-02-15 18:31:26+00:00,2,0,0,200.00000,byplayer,0,...,8.0,18.0,steam,801.0,official,Bronze,both,True,False,False
1,fcf230a3-e9bd-4c7e-a5ac-28e998a499d5,Desert_Main,squad,2026-02-19 06:03:34+00:00,2,0,3,203.72414,byplayer,2,...,9.0,10.0,steam,1910.0,competitive,Gold,both,True,False,False
2,d76c3295-4742-467e-9d86-c58edfebcdcf,Neon_Main,squad,2026-02-11 20:02:02+00:00,1,4,6,451.97598,byplayer,2,...,10.0,7.0,steam,0.0,official,Unranked,both,True,False,False
3,58878e90-c3c3-4be9-bd42-268f3a7ee3e2,Tiger_Main,squad,2026-02-11 22:08:41+00:00,2,1,5,382.21080,byplayer,0,...,13.0,4.0,steam,0.0,official,Unranked,both,True,False,False
4,4bad3d1a-2bf0-4921-8b14-50984cbdf2eb,Baltic_Main,squad,2026-02-16 14:36:19+00:00,3,0,7,428.00513,byplayer,1,...,16.0,8.0,steam,0.0,official,Unranked,both,True,False,False
